# Deep Learning

# Tutorial 6: Linear Regression

In this tutorial, we will cover:

- Linear Regression: Data creation, model setup, training

Prerequisites:

- Python, Tensor basics

Our contacts:

- Niklas Beuter (niklas.beuter@th-luebeck.de)
- Fenja Falta (fenja.falta@th-luebeck.de)

Course:

- Slides and notebooks will be available at https://lernraum.th-luebeck.de/course/view.php?id=5383


# Linear Regression

Linear regression is one of the most basic and widely used statistical and machine learning methods. It is used to model a linear relationship between a dependent variable \(y\) and one or more independent variables \(X\).

## Fundamental Principle

The goal of linear regression is to find a line (or a hyperplane in higher dimensions) that best approximates the data points. The equation for simple linear regression with one independent variable is as follows:

$$ y = w_0x + b + \epsilon $$

- \(y\) is the dependent variable.
- \(x\) is the independent variable.
- \($b$\) is the y-intercept of the regression line.
- \($w_0$\) is the slope of the regression line.
- \($\epsilon$\) represents the error term or residuals, which are the difference between the observed values and the values predicted by the model.

## Areas of Application

Linear regression finds wide application across many fields including economics, engineering, natural sciences, and social sciences to investigate relationships between variables and to make predictions.

## Least Squares Method

The parameters \($w_0$\) and \($b$\) are commonly estimated using the least squares method, where the sum of the squared residuals is minimized to achieve the best possible fit to the data.

## Assumptions

For linear regression to be applied, certain assumptions need to be met including linearity, homoscedasticity, independence of errors, no or little multicollinearity, and normal distribution of errors.


# Going into the code

First, we need to install some dependencies.

In [ ]:
!pip install matplotlib numpy pandas torch

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

## Linear Regression example

Next, we are going to implement and use linear regression ourself. First, we create an own dataset. Here, we create a dataset for car price prediction using mileage and age.

In [ ]:
# First, we create our own dataset. In this case we want to predict the value of cars

# Set seed for reproducibility
np.random.seed(42)

# Generate random data
num_samples = 1000
kilometer = np.random.uniform(0, 200000, num_samples)  # Kilometerstand zwischen 0 und 200.000 km
age = np.random.uniform(0, 15, num_samples)  # Alter zwischen 0 und 15 Jahren

# Define parameters for the linear model
base_price = 30000  # Basispreis eines neuen Fahrzeugs
price_decline_per_km = 0.05  # Preisverfall pro Kilometer
price_decline_per_year = 1500  # Preisverfall pro Jahr

# Calculate vehicle price with some added noise for variability
noise = np.random.normal(0, 1000, num_samples)  # Zufällige Streuung
price = base_price - (kilometer * price_decline_per_km + age * price_decline_per_year) + noise

# Create DataFrame
data = pd.DataFrame({
    'Kilometerstand': kilometer,
    'Alter': age,
    'Preis': price
})

data.head()


In [ ]:
data["Preis"][0]

In [ ]:
# plot the created dataset

kilometers = data["Kilometerstand"][:]
age = data["Alter"][:]
prices = data["Preis"][:]

# Plot for kilometers vs price
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.scatter(kilometers, prices, color='blue')
plt.scatter(40000, 18000, color='green')
plt.title('Kilometers vs Price')
plt.xlabel('Kilometers')
plt.ylabel('Price')
plt.grid(True)

# Plot for age vs price
plt.subplot(1, 2, 2)
plt.scatter(age, prices, color='red')
plt.scatter(6, 16000, color='green')
plt.title('Age vs Price')
plt.xlabel('Age')
plt.ylabel('Price')
plt.grid(True)

# Displaying the plots
plt.tight_layout()
plt.show()

### Linear model and loss and Optimization via gradient descent

Our linear regression model is $ y = f(x) = w \cdot x + b$. For an easy start we just use the 'Alter' data for predicting a price.

We solve for $w$ and $b$ using gradient descent (SGD = stochastic gradient descent). Initialize parameters $w$ and $b$ randomly. The learning rate is set to $0.001$. The number of epochs are the iterations we use for optimizing our parameters.

The loss function is just the squared mean between the predicted value $\hat{Y}$ and the real $Y$ value $L = mean(\hat{Y} - Y)^2$.

So, what we are doing here is the following. We create a starting point for our linear regression model by giving the parameters some kind of starting value (here a value between 0 and 1). Then, we iterate a learning cycle, where we pass the data into the model and predict an output. Of course, the output is not fitting to the real value as we guessed the starting point. The difference to the known output from the training data helps us to calculate the loss. This scalar value is used in the optimizer step to calculate gradients for all parameters. We use this information to update all parameters. The trick is to reduce the update step by a factor (the learning rate) and just update all parameters slightly. Doing this many iterations slightly moves the parameters to optimized values (stochastic gradient descent).

In [ ]:
# Convert the DataFrame to PyTorch tensors
X = torch.tensor(data['Alter'].values, dtype=torch.float32).view(-1, 1)  # Reshaping for matrix operations
Y = torch.tensor(data['Preis'].values, dtype=torch.float32).view(-1, 1)

# Initialize model parameters w and b
w = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

# Define the optimizer
optimizer = torch.optim.SGD([w, b], lr=0.001)

# Training loop
num_epochs = 10000
for epoch in range(num_epochs):
    # Forward pass: Compute predicted Y by passing X to the model. We use matrix multiplication here for speed reasons.
    # Linear model equation: Y_pred = X * w + b (see 04_tensor_operations, @ overloaded matrix multiplication)
    Y_pred = X @ w + b

    # Compute loss: Mean Squared Error (MSE)
    loss = torch.mean((Y_pred - Y) ** 2)

    # Zero gradients -> Is always done before each iteration to reset all gradient calculations from the last step
    # perform a backward pass -> computes the gradient of the loss for every parameter that has requires_grad=True.
    # and update the weights -> updates the parameters based on the current gradients.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print loss every 1000 epochs
    if epoch % 1000 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

# Print final parameters
print(f'w: {w.item()}, b: {b.item()}')


Have a look at the loss value between each iteration. This value should decrease as the difference between the predictions and real values should get less. Observing this value is crucial for deep learning as we can see, if our parameter selection (initialization, learning rate) is good and leads to a good model. We will come back to this, what it means, if this value is not decreasing or even raising.

At the end, you should always do a visualization of your result, here the linear fit.

In [ ]:
# Visualization
plt.figure(figsize=(10, 6))
plt.scatter(X.numpy(), Y.numpy(), color='blue', label='Actual data', alpha=0.5)
plt.plot(X.numpy(), (X @ w + b).detach().numpy(), color='red', label='Regression Line')
plt.xlabel('Alter')
plt.ylabel('Preis')
plt.title('Linear Regression Model')
plt.legend()
plt.show()

### Connection of Loss and Optimizer

You might have wondered how the optimizer knows about the loss and the gradients. The cool thing is the optimizer in PyTorch doesn't need to be explicitly told about the loss during its initialization. Instead, it works through the connection established by the computational graph that PyTorch builds during the forward pass and the subsequent .backward() call on the loss tensor.

In deep learning you are always doing the following steps, which establish the connection between loss and optimizer:

* **Forward Pass**: You compute the predicted values (Y_pred) based on your model's parameters (w and b in this case) and inputs (X). This operation, along with the loss computation that follows, constructs a computational graph. This graph contains information about which operations were performed on the tensors and how they are connected.

* **Compute Loss**: The loss is computed using the predictions and the actual values (Y). This computation also becomes part of the computational graph. The loss tensor directly or indirectly references the model parameters that contributed to its value.

* **Backward Pass**: When loss.backward() is called, PyTorch traverses this graph starting from the loss tensor backward to all tensors that have requires_grad=True set and computes the gradient of the loss with respect to these tensors. This is where the connection between the loss and the optimizer's parameters is made use of. The gradients are stored in the .grad attribute of each tensor (parameter) involved in the computation of the loss.

* **Optimizer Step**: The optimizer is initialized with a list of parameters ([w, b] in this example). After the backward pass, each of these parameters has its .grad attribute populated with the gradients computed by .backward(). When optimizer.step() is called, it updates each parameter by taking a step in the direction that minimally decreases the loss, using the gradients stored in .grad. This step is determined by the optimization algorithm the optimizer implements (SGD in this case) and the learning rate.

### Prediction of new values

Now we can start predicting a price for other cars using the trained model.

In [ ]:
# New data for prediction
new_alter = torch.tensor([15.0])  # For example, predicting price for a car of age 10 years

# Predicting the price using the trained model
predicted_price = new_alter * w.detach() + b.detach()

print(f'Predicted Price: {predicted_price.item()}€')


## Multiple linear regression

In multiple linear regression, each input feature has its own coefficient (weight), and the model learns the relationship between each feature and the target variable.

The linear model equation for multiple linear regression, considering both "Alter" and "Kilometerstand" as input features, would be:

$ Preis = w_0 * Alter + w_1 * Kilometerstand + b $

As the input data is not equally scaled (Kilometer vs Age) the model would not learn a good prediction. We need to bring it to a similar value range. This is done by `normalization` using a subtraction of the mean and a division by the standard deviation.

see: [Normalization - Wikipedia](https://en.wikipedia.org/wiki/Normalization_(statistics))

In [ ]:
# Assuming 'data' is your DataFrame
# Convert the DataFrame to PyTorch tensors. This time, include both 'Alter' and 'Kilometerstand'.
X = torch.tensor(data[['Alter', 'Kilometerstand']].values, dtype=torch.float32)
Y = torch.tensor(data['Preis'].values, dtype=torch.float32).view(-1, 1)

# Normalize features
X_norm = (X - X.mean(dim=0)) / X.std(dim=0)

# Initialize model parameters w for both features and b
w = torch.randn(2, 1, requires_grad=True)  # Now w has two weights: one for 'Alter' and one for 'Kilometerstand'
b = torch.randn(1, requires_grad=True)

# Define the optimizer
optimizer = torch.optim.SGD([w, b], lr=1e-4)  # Adjust the learning rate if necessary

# Training loop
num_epochs = 25000
for epoch in range(num_epochs):
    # Forward pass: Compute predicted Y by passing X to the model
    Y_pred = X_norm @ w + b  # Matrix multiplication to handle multiple features

    # Compute loss: Mean Squared Error (MSE)
    loss = torch.mean((Y_pred - Y) ** 2)

    # Zero gradients, perform a backward pass, and update the weights.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Print loss every 1000 epochs
    if epoch % 1000 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

# Print final parameters
print(f'w: {w.detach().view(-1).numpy()}, b: {b.detach().item()}')


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # Import the 3D plotting module

# Assuming w and b are your model's parameters after training
# And X and Y are your feature matrix and target vector

# Prepare data for plotting
X_plot = X_norm[:, 0].numpy()  # Alter
Y_plot = X_norm[:, 1].numpy()  # Kilometerstand
Z_plot = Y.numpy()        # Preis (actual prices)

# Predicted prices for plotting the regression plane
Z_pred = (X_norm @ w + b).detach().numpy()

# Creating a 3D plot
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Plot actual data points
ax.scatter(X_plot, Y_plot, Z_plot, color='b', label='Actual data')

# Create a mesh grid for the plane
X_grid, Y_grid = np.meshgrid(np.linspace(X_plot.min(), X_plot.max(), 20),
                             np.linspace(Y_plot.min(), Y_plot.max(), 20))

# Predict Z values (prices) on the mesh grid
Z_grid = (w[0].item() * X_grid + w[1].item() * Y_grid + b.item())

# Plot the regression plane
ax.plot_surface(X_grid, Y_grid, Z_grid, color='r', alpha=0.5)

ax.set_xlabel('Alter')
ax.set_ylabel('Kilometerstand')
ax.set_zlabel('Preis')
ax.legend()

plt.show()

Now we can start predicting a price for other cars using the trained model.

In [ ]:
# Assuming 'X_mean' and 'X_std' are the mean and standard deviation of the training data, used for normalization

# New data for prediction (e.g., 'Alter' and 'Kilometerstand')
new_data = torch.tensor([[10.0, 150000.0]])  # For example, a 10-year-old car with 50,000 km

# Normalize new data if the model was trained with normalized features
new_data_norm = (new_data - X.mean(dim=0)) / X.std(dim=0)

# Predicting the price using the trained model
predicted_price = new_data_norm @ w.detach() + b.detach()

print(f'Predicted Price: {predicted_price.item()}€')

# References

This notebook is adapted from or uses following sources:
Markus Enzweiler, markus.enzweiler@hs-esslingen.de, https://github.com/menzHSE/cv-ml-lecture-notebooks.git